# Week 65, Day 4: Smoothing and learning linear dynamical systems

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w065/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The RTS smoother

Complete the backward recursion of rts_smoother, returning smoothed means, covariances and lagged covariances. Compare filtered and smoothed accuracy on a two-dimensional rotating state.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
g0 = np.random.default_rng(654)
T, d, p = 200, 2, 6
th = 0.08
A = 0.99 * np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
C = g0.normal(size=(p, d))
Q = 0.02 * np.eye(d); R = np.diag(g0.uniform(0.3, 1.0, p))
z = np.zeros((T, d)); z[0] = [1.5, 0]
y = np.zeros((T, p))
for t in range(T):
    if t: z[t] = A @ z[t - 1] + g0.multivariate_normal(np.zeros(d), Q)
    y[t] = C @ z[t] + g0.multivariate_normal(np.zeros(p), R)

def kalman_filter(y, A, C, Q, R, m0, P0):
    T_, d_ = len(y), len(m0)
    ms, Ps, mp, Pp = [], [], [], []
    m, P = m0.copy(), P0.copy()
    for t in range(T_):
        if t:
            m, P = A @ m, A @ P @ A.T + Q
        mp.append(m.copy()); Pp.append(P.copy())
        S = C @ P @ C.T + R
        K = P @ C.T @ np.linalg.inv(S)
        m = m + K @ (y[t] - C @ m); P = (np.eye(d_) - K @ C) @ P
        ms.append(m.copy()); Ps.append(P.copy())
    return map(np.array, (ms, Ps, mp, Pp))

def rts_smoother(y, A, C, Q, R, m0, P0):
    ms, Ps, mp, Pp = kalman_filter(y, A, C, Q, R, m0, P0)
    T_ = len(y)
    mhat, Phat = ms.copy(), Ps.copy()
    Plag = np.zeros_like(Ps)
    for t in range(T_ - 2, -1, -1):
        J = Ps[t] @ A.T @ np.linalg.inv(Pp[t + 1])
        mhat[t] = ...
        Phat[t] = ...
        Plag[t + 1] = Phat[t + 1] @ J.T
    return mhat, Phat, Plag, ms, Ps

ms_s, Ps_s, Plag, ms_f, Ps_f = rts_smoother(y, A, C, Q, R, np.zeros(d), np.eye(d))
err_f = np.sqrt(np.mean((ms_f - z) ** 2)); err_s = np.sqrt(np.mean((ms_s - z) ** 2))
print(f"error: filtered {err_f:.4f}, smoothed {err_s:.4f}")
print(f"mean posterior variance: filtered {np.mean(np.trace(Ps_f, axis1=1, axis2=2)):.4f}, smoothed {np.mean(np.trace(Ps_s, axis1=1, axis2=2)):.4f}")

In [ ]:
import numpy as np
def test_better():
    "smoothing is more accurate than filtering, and its covariances are smaller"
    assert err_s < err_f and np.all(np.trace(Ps_s, axis1=1, axis2=2) <= np.trace(Ps_f, axis1=1, axis2=2) + 1e-9)
def test_ends():
    "at the last time step, smoothed and filtered estimates agree"
    assert np.allclose(ms_s[-1], ms_f[-1])

test_better()
test_ends()
print("All checks passed")

### Exercise: EM for a linear dynamical system

Complete the M step for A and Q from the smoothed moments, and run 100 iterations of EM from a random start to recover the dynamics of the simulated population.

In [ ]:
def log_lik(y, A, C, Q, R, m0, P0):
    from scipy.stats import multivariate_normal as mvn
    m, P, ll = m0.copy(), P0.copy(), 0.0
    for t in range(len(y)):
        if t: m, P = A @ m, A @ P @ A.T + Q
        S = C @ P @ C.T + R
        ll += mvn.logpdf(y[t] - C @ m, np.zeros(len(S)), S)
        K = P @ C.T @ np.linalg.inv(S)
        m = m + K @ (y[t] - C @ m); P = (np.eye(len(m)) - K @ C) @ P
    return ll

ge = np.random.default_rng(6541)
A_hat = 0.9 * np.eye(d) + 0.1 * ge.normal(size=(d, d))
C_hat = ge.normal(size=(p, d)); Q_hat = 0.1 * np.eye(d); R_hat = np.eye(p)
m0, P0 = np.zeros(d), np.eye(d)
ll_lds = []
for it in range(100):
    mh, Ph, Plg, _, _ = rts_smoother(y, A_hat, C_hat, Q_hat, R_hat, m0, P0)
    ll_lds.append(log_lik(y, A_hat, C_hat, Q_hat, R_hat, m0, P0))
    Ezz = Ph + np.einsum("ti,tj->tij", mh, mh)
    Ezz1 = Plg[1:] + np.einsum("ti,tj->tij", mh[1:], mh[:-1])
    S11, S10, S00 = Ezz[1:].sum(0), Ezz1.sum(0), Ezz[:-1].sum(0)
    A_hat = ...
    Q_hat = ...
    C_hat = (y.T @ mh) @ np.linalg.inv(Ezz.sum(0))
    R_hat = np.diag(np.diag((y.T @ y - C_hat @ (y.T @ mh).T) / len(y)))
    m0, P0 = mh[0], Ph[0]
print(f"log-likelihood {ll_lds[0]:.1f} -> {ll_lds[-1]:.1f}")
print("eigenvalues of A: learned", np.round(np.linalg.eigvals(A_hat), 3), " true", np.round(np.linalg.eigvals(A), 3))

In [ ]:
import numpy as np
def test_up():
    "the log-likelihood never decreases"
    assert all(b >= a - 1e-6 for a, b in zip(ll_lds, ll_lds[1:]))
def test_eigs():
    "the learned dynamics have the right eigenvalues (rotation and decay) to within 10%"
    ev = np.sort_complex(np.linalg.eigvals(A_hat))
    tru = np.sort_complex(np.linalg.eigvals(A))
    assert np.allclose(np.abs(ev), np.abs(tru), rtol=0.1) and np.allclose(np.sort(np.abs(np.angle(ev))), np.sort(np.abs(np.angle(tru))), atol=0.03)

test_up()
test_eigs()
print("All checks passed")

### Exercise: Predicting held-out neurons

Hold out two of the six neurons. Fit the LDS on the other four, infer the latent trajectory from them, then fit the held-out neurons' loadings by regression on the inferred latents and report their R squared. This is the co-smoothing test.

In [ ]:
held = [4, 5]
keep = [0, 1, 2, 3]
y_keep, y_held = y[:, keep], y[:, held]
A_k = 0.9 * np.eye(d) + 0.1 * ge.normal(size=(d, d))
C_k = ge.normal(size=(len(keep), d)); Q_k = 0.1 * np.eye(d); R_k = np.eye(len(keep))
m0k, P0k = np.zeros(d), np.eye(d)
for it in range(100):
    mh, Ph, Plg, _, _ = rts_smoother(y_keep, A_k, C_k, Q_k, R_k, m0k, P0k)
    Ezz = Ph + np.einsum("ti,tj->tij", mh, mh)
    Ezz1 = Plg[1:] + np.einsum("ti,tj->tij", mh[1:], mh[:-1])
    S11, S10, S00 = Ezz[1:].sum(0), Ezz1.sum(0), Ezz[:-1].sum(0)
    A_k = S10 @ np.linalg.inv(S00); Q_k = (S11 - A_k @ S10.T) / (len(y) - 1)
    C_k = (y_keep.T @ mh) @ np.linalg.inv(Ezz.sum(0))
    R_k = np.diag(np.diag((y_keep.T @ y_keep - C_k @ (y_keep.T @ mh).T) / len(y)))
    m0k, P0k = mh[0], Ph[0]
tr, te = np.arange(140), np.arange(140, 200)
W_held = np.linalg.lstsq(mh[tr], y_held[tr], rcond=None)[0]
pred = mh[te] @ W_held
r2_held = ...
print(f"held-out neurons, test R squared from the inferred latents: {r2_held:.3f}")

In [ ]:
import numpy as np
def test_cosmooth():
    "the fitted LDS predicts held-out neurons better than their mean"
    assert r2_held > 0.3

test_cosmooth()
print("All checks passed")

### Exercise: A change of coordinates

Apply a random invertible transformation M to the fitted model and check that the log-likelihood and the eigenvalues of A are unchanged while the matrices themselves are not.

In [ ]:
M = np.array([[1.4, 0.7], [-0.3, 0.9]])
A_M = ...
C_M = C_hat @ np.linalg.inv(M)
Q_M = M @ Q_hat @ M.T
ll_plain = log_lik(y, A_hat, C_hat, Q_hat, R_hat, m0, P0)
ll_M = log_lik(y, A_M, C_M, Q_M, R_hat, M @ m0, M @ P0 @ M.T)
print(f"log-likelihood: original {ll_plain:.6f}, transformed {ll_M:.6f}")
print("A original:\n", np.round(A_hat, 3), "\nA transformed:\n", np.round(A_M, 3))
print("eigenvalues:", np.round(np.linalg.eigvals(A_hat), 4), np.round(np.linalg.eigvals(A_M), 4))

In [ ]:
import numpy as np
def test_same_ll():
    "the transformed model has the same log-likelihood"
    assert abs(ll_M - ll_plain) < 1e-6
def test_eigs_same():
    "its dynamics matrix differs entrywise but has the same eigenvalues"
    assert not np.allclose(A_M, A_hat, atol=1e-3) and np.allclose(np.sort_complex(np.linalg.eigvals(A_M)), np.sort_complex(np.linalg.eigvals(A_hat)))

test_same_ll()
test_eigs_same()
print("All checks passed")